# Análise Preditiva — Nascidos Vivos Brasileiros (SINASC/DataSUS 2019–2023)

Este notebook importa o módulo **`ml/pipeline.py`** — onde mora toda a lógica de limpeza, preparação, modelagem e avaliação — e apresenta os resultados com gráficos e narrativa.

**Problemas de ML abordados:**
1. **Classificação — Baixo peso ao nascer** (`BAIXO_PESO` < 2500 g) — ~1,8% dos casos
2. **Classificação — Prematuridade** (`PREMATURO` < 37 semanas) — ~10% dos casos
3. **Regressão — Peso ao nascer** (`PESO_GRAMAS`)

> ⚠️ **Anti-vazamento:** `PESO_GRAMAS` define `BAIXO_PESO` e `SEMANAS_GESTACAO` define `PREMATURO`. Essas variáveis **nunca** entram como features do próprio alvo. `APGAR5` e `TIPO_PARTO` só são conhecidos após o parto e ficam de fora da predição antecipada.

In [ ]:
import os, sys
# Torna o import de pipeline.py robusto (notebook aberto de ml/ ou da raiz do repositório)
sys.path.insert(0, os.path.join(os.getcwd(), 'ml') if os.path.basename(os.getcwd()) != 'ml' else os.getcwd())
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import pipeline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 4.5)

## 1. Carregamento e limpeza

Replicação fiel da limpeza documentada no `data-base-analysis.Rmd` (SEXO, PESO_GRAMAS, IDADE_MAE, UF e duplicatas).

In [ ]:
nasc, uf = pipeline.load_data()
df, stats = pipeline.clean_data(nasc, uf)
df = pipeline.adicionar_regiao(pipeline.derive_features(df), uf)
print(f'Iniciais: {stats["inicial"]:,} | Finais: {len(df):,} | Removidos: {stats["removidos"]:,} ({stats["removidos"]/stats["inicial"]*100:.1f}%)')
pd.DataFrame(stats['etapas'])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cores = ['#4c72b0', '#c44e52']
for ax, col, titulo in zip(axes[:2], ['BAIXO_PESO', 'PREMATURO'], ['Baixo peso (<2500g)', 'Prematuridade (<37 sem)']):
    df[col].replace({0: 'Não', 1: 'Sim'}).value_counts().sort_index().plot.bar(ax=ax, color=cores)
    ax.set_title(f'{titulo} — {df[col].mean()*100:.1f}% positivos')
    ax.set_ylabel('Nascimentos')
    ax.tick_params(axis='x', rotation=0)
df.groupby('REGIAO')['BAIXO_PESO'].mean().mul(100).sort_values().plot.bar(ax=axes[2], color='#55a868')
axes[2].set_title('% baixo peso por região')
axes[2].set_ylabel('%')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 2. Modelagem preditiva — validação cruzada estratificada (5 dobras)

Modelos comparados: **Dummy (piso)**, **Regressão Logística** (baseline interpretável), **Random Forest** (controle) e **XGBoost** (modelo principal da literatura). Métricas reportadas com média ± desvio nas 5 dobras.

A curva **Precision–Recall** é a métrica principal por causa do desbalanceamento das classes. Todas as curvas e o limiar usam previsões **out-of-fold** (honestas).

In [ ]:
def rodar_tarefa_classificacao(df, target, features, slug):
    X, y = pipeline.prepare_model_data(df, target, features)
    print(f'--- {slug} | alvo {target} | n={len(X):,} | positivos={y.mean()*100:.1f}%')
    models = pipeline.build_models(y)
    res = pipeline.cross_validate_classifiers(X, y, models)
    display(pipeline.tabela_metricas(res))
    best, linha = pipeline.melhor_modelo(res)
    print(f'Melhor (PR-AUC): {best} — {linha["pr_auc_media"]:.3f} ± {linha["pr_auc_std"]:.3f}')
    oof = pipeline.oof_probas(models, X, y)  # OOF calculado UMA vez por modelo
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
    pipeline.plot_roc_curves(a1, X, y, models, oof=oof)
    pipeline.plot_pr_curves(a2, X, y, models, oof=oof)
    plt.tight_layout()
    plt.show()
    proba = oof[best]
    thr = pipeline.optimal_threshold(y, proba)
    pred = (proba >= thr).astype(int)
    print(f'Limiar ótimo (Youden): {thr:.3f} → Sens. {pipeline.sensibilidade(y, pred):.3f} | Esp. {pipeline.especificidade(y, pred):.3f} | F1 {pipeline.f1(y, pred):.3f}')
    fig, ax = plt.subplots(figsize=(5, 4))
    pipeline.plot_confusion_matrix(ax, y, pred, ['Não', 'Sim'], f'{target} — OOF (limiar {thr:.2f})')
    plt.show()
    return X, y, models, res, best, thr

In [ ]:
# Tarefa 1 — Baixo peso ao nascer (sem PESO_GRAMAS como feature!)
X1, y1, models1, res1, best1, thr1 = rodar_tarefa_classificacao(
    df, 'BAIXO_PESO', ['IDADE_MAE', 'CONSULTAS_PRENATAL', 'SEMANAS_GESTACAO', 'SEXO', 'REGIAO', 'ANO'], 'baixo_peso')

In [ ]:
# Tarefa 2 — Prematuridade (sem SEMANAS_GESTACAO como feature!)
X2, y2, models2, res2, best2, thr2 = rodar_tarefa_classificacao(
    df, 'PREMATURO', ['IDADE_MAE', 'CONSULTAS_PRENATAL', 'SEXO', 'REGIAO', 'ANO'], 'prematuro')

## 3. Regressão — Peso ao nascer

Estimativa do `PESO_GRAMAS` (alvo contínuo) com Random Forest e XGBoost regressores, em 5-fold CV.

In [ ]:
Xr, yr = pipeline.prepare_model_data(df, 'PESO_GRAMAS', ['IDADE_MAE', 'CONSULTAS_PRENATAL', 'SEMANAS_GESTACAO', 'SEXO', 'REGIAO', 'ANO'])
print(f'Regressão PESO_GRAMAS — n={len(Xr):,} | média={yr.mean():.0f} g')
res_reg = pipeline.cross_validate_regressors(Xr, yr, pipeline.build_models(yr, 'regression'))
display(pipeline.tabela_metricas_regressao(res_reg))
fig, ax = plt.subplots(figsize=(6.5, 6))
pipeline.plot_scatter_predicoes(ax, Xr, yr, pipeline.build_models(yr, 'regression')['XGBoost Regressor'])
plt.show()

## 4. Interpretação com SHAP

O SHAP explica **por que** o modelo fez cada predição e quais fatores mais influenciam o risco — essencial em saúde pública.

In [ ]:
# SHAP — Baixo peso (melhor modelo)
best_model1 = pipeline.build_models(y1)[best1].fit(X1, y1)
_ = pipeline.plot_shap_summary(best_model1, X1, list(X1.columns))
plt.title(f'SHAP — BAIXO_PESO ({best1})')
plt.show()

In [ ]:
# SHAP — Prematuridade (melhor modelo)
best_model2 = pipeline.build_models(y2)[best2].fit(X2, y2)
_ = pipeline.plot_shap_summary(best_model2, X2, list(X2.columns))
plt.title(f'SHAP — PREMATURO ({best2})')
plt.show()

## 5. Conclusão e insights

- **A Regressão Logística venceu as duas tarefas** (PR-AUC 0.125 em `BAIXO_PESO` e 0.121 em `PREMATURO`). O sinal preditivo é majoritariamente linear neste dataset; em `BAIXO_PESO` a variável `SEMANAS_GESTACAO` carrega quase todo o poder discriminativo (ROC-AUC 0.785). Para este projeto, o modelo recomendado é **Regressão Logística + SHAP** — performance equivalente à das árvores, com interpretabilidade muito superior.
- **Prematuridade é difícil de prever sem a idade gestacional** (ROC-AUC ~0.53, pouco acima do piso): sem `SEMANAS_GESTACAO` (que define o alvo), as características maternas isoladas explicam pouco. Este é um achado honesto e esperado — não um bug.
- O **Dummy (piso)** mostra o patamar ingênuo: PR-AUC 0.018 em `BAIXO_PESO` (prevalência de 1,8%). Todo ganho acima disso é valor real.
- O **limiar de Youden** (previsões out-of-fold) ajusta o trade-off sensibilidade × especificidade — uma decisão de saúde pública: em `BAIXO_PESO`, priorizar sensibilidade (0.674) captura mais casos de risco em troca de mais alarmes falsos.
- Na **regressão**, o XGBoost (MAE ≈ 279 g, R² ≈ 0.15) superou o Random Forest, mas o erro é grande para uso clínico individual — útil apenas como estimativa populacional.
- Para regenerar todos os artefatos em **`ml/resultados/`** (CSVs + PNGs) sem o notebook: `python ml/pipeline.py`.

In [ ]:
# Comparativo final das duas tarefas
final = pd.concat([
    pipeline.tabela_metricas(res1).assign(tarefa='baixo_peso'),
    pipeline.tabela_metricas(res2).assign(tarefa='prematuro'),
], ignore_index=True)
final[['tarefa', 'modelo', 'roc_auc', 'pr_auc', 'f1', 'sensibilidade', 'especificidade']]